# 14. The builder: exercises

Exercises of [chapter 14](../../14_builder.md). Run the setup cell first.

Work in a copy of this notebook, under [`docs/manual/solutions/`](..), in a folder `solutions_<you>`: it is regenerated by [`docs/manual/tools/build_notebooks.py`](../../tools/build_notebooks.py), and its links are written for that folder.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import copy, datetime, inspect, json, shutil, tempfile
import geopandas as gpd
import yaml
from shapely.geometry import Point
from data_acquisition.catalog import build, stac
from data_acquisition.catalog import summary as summaries
from data_acquisition.readers import RECORDS, geotiff, photos, vector

REFERENCE = yaml.safe_load((ROOT / "docs/manual/solutions/training_pond.yaml").read_text())


def load(s):
    path = Path(tempfile.mkdtemp()) / "summary.yaml"
    path.write_text(yaml.safe_dump(copy.deepcopy(s), sort_keys=False))
    return summaries.load(str(path))

**1. Trace one entry.** Take the DTM entry of the reference summary: resolve it with its group, compute its time and its item id, then build it with `build.data_item` and print where each value came from (the `beaver:` fields' sources, the provenance, the flags).

> **From the chapter.** §1: the steps of `build`, and §2: one entry becoming one item.
>
> **Code.** `summaries.resolve` and `summaries.epoch` in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py), and `build.data_item` in [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py). `load(s)` is defined above.

In [ ]:
s = load(REFERENCE)
group = s["groups"][0]
entry = next(f for f in group["files"] if f["path"] == "dem_2022.asc")
...  # your code here (summaries.resolve, summaries.epoch, build.data_item)

**2. Dates.** Compute `summary.epoch` for a day, a month, a year, a range and an `unknown` date; and `build._period` for a record whose dates are `['2001', '2024-06']`.

> **From the chapter.** §3, on time: precision, labels, and a record's dates.
>
> **Code.** `summaries.epoch` in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py), and `build._period` in [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py).

In [ ]:
...  # your code here

**3. Bands.** Call `build._bands` for the sample's orthomosaic with no bands given, with the right ones, with three bands, and with the order reversed; then for a DSM.

> **From the chapter.** §3, on bands: the summary's bands, a metric band name, the file's own, or `b1`, `b2`.
>
> **Code.** `build._bands` in [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py), and `geotiff.info` in [`data_acquisition/readers/geotiff.py`](../../../../data_acquisition/readers/geotiff.py).

In [ ]:
header = geotiff.info(DATASET / "ortho_2023-07-12.tif")
...  # your code here (build._bands(product, wanted, header, entity=None))

**4. One item per layer.** In a scratch copy of the sample, write a GeoPackage with layers `dams_2021` and `dams_2023` (and one called `notes`), describe it (product `annotations`, variant `by_year`) with `layers: {pattern: '_(\d{4})$', date_precision: year}`, and build. First date each layer name with `summaries.layer_epoch` ([chapter 9](../../09_describing_a_dataset.md)): which gives None? Then: how many items, and what does the report say?

> **From the chapter.** §1, the table of entries, and [chapter 9](../../09_describing_a_dataset.md), §5 on `layer_epoch`.
>
> **Code.** `summaries.layer_epoch` in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py), and `build.build` in [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py).

In [ ]:
scratch = Path(tempfile.mkdtemp()) / "root"
shutil.copytree(SAMPLE, scratch, ignore=shutil.ignore_patterns("stac"))
...  # your code here

**5. Attachments.** Give README.txt a date (`2023-07-12`) in a copy of the summary, then `attach_to: [chm]`. Where does it attach each time?

> **From the chapter.** §4: how a report or metadata file finds its items.
>
> **Code.** `build.build`, in [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py). The attachments are in each item's assets, and in the report.

In [ ]:
...  # your code here

**6. Find it in the code.** Where is the item id made, and where is a collection's licence set to `various`?

> **From the chapter.** §2 for the id, §4 for the collection's licence.
>
> **Code.** Look in [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py) (imported above as `build`).

In [ ]:
...  # your code here (inspect)

**7. Places and sizes.** For the sample's orthomosaic and the extract's chunk: `stac.raster_crs`, `stac.footprint`, `stac.resolution_m`. Then `resolution_m` for a raster in degrees (a copy of the header with `epsg` 4326 and pixels of 0.000001°). What is `stac.lonlat_box` for a box in UTM? What does `stac.size_of` give for a file and for a folder? And `stac.multihash` for an MD5, a SHA-256 and a SHA-1?

> **From the chapter.** §5: the table of `stac.py`'s functions.
>
> **Code.** `stac.raster_crs`, `stac.footprint`, `stac.resolution_m`, `stac.lonlat_box`, `stac.size_of` and `stac.multihash`, in [`data_acquisition/catalog/stac.py`](../../../../data_acquisition/catalog/stac.py); `geotiff.info` in [`data_acquisition/readers/geotiff.py`](../../../../data_acquisition/readers/geotiff.py).

In [ ]:
ortho = geotiff.info(DATASET / "ortho_2023-07-12.tif")
chunk = geotiff.info(EXTRACT / "2018-07-26/orthomosaic_chunk.tif")
...  # your code here

**8. Items by hand.** Make items directly: `stac.raster_item` for the orthomosaic with one date and with a span; for the extract's chunk (no EPSG code); `stac.vector_item` for the dams; `stac.photos_item` for the photos. Which STAC fields does each write? Add the README with `stac.add_file_asset`, and find its path with `stac.asset_path`.

> **From the chapter.** §5: `stac.py` writes the STAC, and knows nothing about summaries.
>
> **Code.** `stac.raster_item`, `stac.vector_item`, `stac.photos_item`, `stac.add_file_asset` and `stac.asset_path`, in [`data_acquisition/catalog/stac.py`](../../../../data_acquisition/catalog/stac.py); the readers `geotiff.info`, `vector.info` and `photos.info` in [`data_acquisition/readers/`](../../../../data_acquisition/readers).

In [ ]:
day, month_end = datetime.date(2023, 7, 12), datetime.date(2023, 7, 31)
ortho = geotiff.info(DATASET / "ortho_2023-07-12.tif")
...  # your code here